# 화장품 성분 데이터 전처리 가이드라인

**데이터 소스:**
- `hwahae_all.csv` — 화해 앱 크롤링 데이터
- `coos_성분정보.xlsx` — Coos 성분 정보
- `paulaschoice_ingredients.csv` — Paula's Choice 성분 정보

**전처리 목표:**
1. 각 소스별 데이터 로드 및 기본 탐색
2. 결측치 / 중복 제거
3. 성분명 정규화 (한글↔영문, 대소문자, 특수문자)
4. 공통 스키마 통합 및 소스 태깅
5. 최종 통합 데이터셋 저장

---
## 0. 환경 설정 및 라이브러리 임포트

In [10]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
import os

# 경고 메시지 최소화
import warnings
warnings.filterwarnings('ignore')

origin_path = os.getcwd()
print(origin_path)
if origin_path != "C:\\team_project\\flow":
    os.chdir("C:\\team_project\\flow")
print(origin_path)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 80)

DATA_DIR = Path(origin_path + '/00_data/00_raw')
OUTPUT_DIR = Path(origin_path + '/00_data/00_raw/output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('✅ 라이브러리 로드 완료: ', DATA_DIR)

C:\python


FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: 'C:\\team_project\\flow'

---
## 1. 데이터 로드

### 📌 가이드라인
- 인코딩 문제(특히 한글)를 위해 `utf-8-sig` 또는 `cp949`를 순차 시도
- xlsx는 시트 이름 확인 후 로드
- 로드 직후 shape, dtypes, 샘플 확인

In [8]:
# ─── 1-1. 화해 데이터 ───────────────────────────────────────────────
def load_csv_safe(filepath, **kwargs):
    """인코딩 자동 감지 CSV 로더"""
    for enc in ['utf-8-sig', 'utf-8', 'cp949', 'euc-kr']:
        try:
            df = pd.read_csv(filepath, encoding=enc, **kwargs)
            print(f'  [{filepath.name}] 인코딩: {enc}, shape: {df.shape}')
            return df
        except (UnicodeDecodeError, Exception):
            continue
    raise ValueError(f'인코딩 자동 감지 실패: {filepath}')

print("파일 존재:", (DATA_DIR / 'hwahae_all.csv').exists())
df_hwahae = load_csv_safe(DATA_DIR / 'hwahae_all.csv')
df_hwahae.head(3)

C:\python
파일 존재: False


ValueError: 인코딩 자동 감지 실패: ..\..\00_data\00_raw\hwahae_all.csv

In [ ]:
# ─── 1-2. Coos 성분 데이터 (xlsx) ────────────────────────────────────
xlsx_path = DATA_DIR / 'coos_성분정보.xlsx'

# 시트 목록 확인
xl = pd.ExcelFile(xlsx_path)
print('시트 목록:', xl.sheet_names)

# 첫 번째 시트 로드 (필요시 sheet_name 변경)
df_coos = pd.read_excel(xlsx_path, sheet_name=xl.sheet_names[0])
print(f'[coos_성분정보.xlsx] shape: {df_coos.shape}')
df_coos.head(3)

In [ ]:
# ─── 1-3. Paula's Choice 데이터 ──────────────────────────────────────
df_paula = load_csv_safe(DATA_DIR / 'paulaschoice_ingredients.csv')
df_paula.head(3)

---
## 2. 기본 탐색 (EDA)

### 📌 가이드라인
- 컬럼명, 결측률, 유니크값 수를 한눈에 확인
- 핵심 컬럼(성분명, 제품명 등) 위주로 분포 파악

In [ ]:
def eda_summary(df, name):
    """데이터프레임 기본 요약"""
    print(f'\n{'='*50}')
    print(f'📊 {name}')
    print(f'{'='*50}')
    print(f'Shape: {df.shape}')
    print(f'\n컬럼 목록:')
    summary = pd.DataFrame({
        'dtype': df.dtypes,
        '결측수': df.isnull().sum(),
        '결측률(%)': (df.isnull().sum() / len(df) * 100).round(2),
        '유니크수': df.nunique()
    })
    print(summary.to_string())
    return summary

eda_hwahae = eda_summary(df_hwahae, '화해')
eda_coos   = eda_summary(df_coos,   'Coos')
eda_paula  = eda_summary(df_paula,  "Paula's Choice")

---
## 3. 결측치 처리

### 📌 가이드라인
| 컬럼 유형 | 처리 방법 |
|---|---|
| 성분명 (핵심 키) | 결측 행 **제거** |
| 설명/효능 텍스트 | `'정보없음'` 또는 빈 문자열로 채움 |
| 수치형 (농도 등) | 중앙값 대체 또는 제거 (도메인 판단) |
| 카테고리 | `'기타'`로 채움 |

In [ ]:
# ─── 아래 컬럼명을 실제 컬럼명으로 수정하세요 ───────────────────────
# 예시: 화해 핵심 컬럼 (실제 컬럼명 확인 후 수정)
HWAHAE_KEY_COLS    = ['성분명']          # 필수 컬럼 (없으면 행 제거)
HWAHAE_FILL_COLS   = ['설명', '효능']    # 텍스트 채움 컬럼

COOS_KEY_COLS      = ['성분명']
COOS_FILL_COLS     = ['설명']

PAULA_KEY_COLS     = ['ingredient_name']
PAULA_FILL_COLS    = ['description', 'function']
# ──────────────────────────────────────────────────────────────────

def handle_missing(df, key_cols, fill_cols=None, fill_value='정보없음'):
    """결측치 처리"""
    before = len(df)
    
    # 핵심 컬럼 결측 행 제거
    existing_key = [c for c in key_cols if c in df.columns]
    if existing_key:
        df = df.dropna(subset=existing_key)
    
    # 텍스트 컬럼 채움
    if fill_cols:
        for col in fill_cols:
            if col in df.columns:
                df[col] = df[col].fillna(fill_value)
    
    print(f'  결측 처리: {before} → {len(df)} 행 ({before-len(df)}행 제거)')
    return df.reset_index(drop=True)

print('화해:')    ; df_hwahae = handle_missing(df_hwahae, HWAHAE_KEY_COLS, HWAHAE_FILL_COLS)
print('Coos:')    ; df_coos   = handle_missing(df_coos,   COOS_KEY_COLS,   COOS_FILL_COLS)
print("Paula's:") ; df_paula  = handle_missing(df_paula,  PAULA_KEY_COLS,  PAULA_FILL_COLS)

---
## 4. 중복 제거

### 📌 가이드라인
- 성분명 기준 완전 중복 제거
- 제품명 + 성분명 조합 중복 제거 (동일 제품-성분 쌍이 여러 번 등장하는 경우)

In [ ]:
def remove_duplicates(df, subset_cols, name):
    """중복 행 제거"""
    before = len(df)
    existing = [c for c in subset_cols if c in df.columns]
    if existing:
        df = df.drop_duplicates(subset=existing, keep='first')
    after = len(df)
    print(f'[{name}] 중복 제거: {before} → {after} 행 ({before-after}행 제거)')
    return df.reset_index(drop=True)

# 실제 컬럼명으로 수정
df_hwahae = remove_duplicates(df_hwahae, ['성분명'],          '화해')
df_coos   = remove_duplicates(df_coos,   ['성분명'],          'Coos')
df_paula  = remove_duplicates(df_paula,  ['ingredient_name'], "Paula's Choice")

---
## 5. 성분명 정규화

### 📌 가이드라인
성분명은 소스마다 표기가 달라 반드시 정규화가 필요합니다:

| 문제 | 예시 | 처리 |
|---|---|---|
| 대소문자 | `Niacinamide` vs `NIACINAMIDE` | 소문자 통일 |
| 앞뒤 공백 | `" 나이아신아마이드 "` | strip() |
| 특수문자 | `Aqua/Water` | 슬래시 앞 성분만 사용 또는 분리 |
| 한글 유니코드 | NFC 정규화 | unicodedata.normalize |
| 괄호 표기 | `Water (Aqua)` | 괄호 제거 |
| INCI명 변형 | `CI 77891` vs `CI77891` | 공백 제거 |

In [ ]:
def normalize_ingredient(name: str) -> str:
    """
    성분명 정규화 함수
    - 유니코드 NFC 정규화 (한글 조합형)
    - 앞뒤 공백 제거 및 소문자 변환
    - 괄호 내용 제거: 'Water (Aqua)' → 'water'
    - 슬래시 앞 첫 성분만 사용: 'Aqua/Water' → 'aqua'
    - 연속 공백 단일 공백으로
    """
    if not isinstance(name, str):
        return ''
    
    # 유니코드 정규화 (한글 NFC)
    name = unicodedata.normalize('NFC', name)
    
    # 앞뒤 공백 제거
    name = name.strip()
    
    # 소문자 변환 (영문 성분명 통일)
    name = name.lower()
    
    # 괄호 내용 제거: '(...)' 또는 '[...]'
    name = re.sub(r'[\(\[\{][^\)\]\}]*[\)\]\}]', '', name)
    
    # 슬래시가 있는 경우 첫 번째 성분만 사용
    # 예: 'aqua/water' → 'aqua'
    # 주의: 한글 성분에는 슬래시가 거의 없으므로 영문만 대상
    if re.search(r'[a-z]', name):  # 영문 포함 성분만
        name = name.split('/')[0]
    
    # 연속 공백 제거
    name = re.sub(r'\s+', ' ', name).strip()
    
    return name


# 정규화 적용 — 실제 성분명 컬럼명으로 수정
def apply_normalization(df, col, new_col='성분명_정규화'):
    if col in df.columns:
        df[new_col] = df[col].apply(normalize_ingredient)
        print(f'  정규화 완료: {col} → {new_col}')
        # 정규화 후 빈 문자열 제거
        before = len(df)
        df = df[df[new_col] != ''].reset_index(drop=True)
        print(f'  빈 값 제거: {before} → {len(df)} 행')
    return df

print('화해:')    ; df_hwahae = apply_normalization(df_hwahae, '성분명')
print('Coos:')    ; df_coos   = apply_normalization(df_coos,   '성분명')
print("Paula's:") ; df_paula  = apply_normalization(df_paula,  'ingredient_name')

# 정규화 결과 샘플 확인
print('\n--- 화해 정규화 샘플 ---')
print(df_hwahae[['성분명', '성분명_정규화']].head(5).to_string())

---
## 6. 컬럼 표준화 및 스키마 통합

### 📌 가이드라인
세 소스를 하나의 공통 스키마로 맞춥니다:

| 통합 컬럼 | 화해 원본 | Coos 원본 | Paula's Choice 원본 |
|---|---|---|---|
| `ingredient_key` | 성분명_정규화 | 성분명_정규화 | 성분명_정규화 |
| `ingredient_name_ko` | 성분명 | 성분명 | _(없음)_ |
| `ingredient_name_en` | _(없으면 None)_ | _(없으면 None)_ | ingredient_name |
| `description` | 설명 | 설명 | description |
| `function` | 효능 | _(없으면 None)_ | function |
| `source` | 'hwahae' | 'coos' | 'paulaschoice' |

In [ ]:
# ─── 실제 컬럼명을 확인하고 매핑 딕셔너리를 수정하세요 ──────────────

# 화해 컬럼 매핑
hwahae_col_map = {
    '성분명_정규화': 'ingredient_key',
    '성분명':        'ingredient_name_ko',
    # '영문명':      'ingredient_name_en',  # 컬럼이 있는 경우 주석 해제
    # '설명':        'description',
    # '효능':        'function',
}

# Coos 컬럼 매핑
coos_col_map = {
    '성분명_정규화': 'ingredient_key',
    '성분명':        'ingredient_name_ko',
    # '설명':        'description',
}

# Paula's Choice 컬럼 매핑
paula_col_map = {
    '성분명_정규화':  'ingredient_key',
    'ingredient_name': 'ingredient_name_en',
    # 'description':   'description',
    # 'function':      'function',
}
# ──────────────────────────────────────────────────────────────────

UNIFIED_COLS = [
    'ingredient_key',
    'ingredient_name_ko',
    'ingredient_name_en',
    'description',
    'function',
    'source'
]

def standardize_schema(df, col_map, source_label):
    """소스별 컬럼을 통합 스키마로 변환"""
    existing_map = {k: v for k, v in col_map.items() if k in df.columns}
    df = df.rename(columns=existing_map)
    df['source'] = source_label
    
    # 없는 컬럼은 NaN으로 추가
    for col in UNIFIED_COLS:
        if col not in df.columns:
            df[col] = np.nan
    
    return df[UNIFIED_COLS]

df_hwahae_std = standardize_schema(df_hwahae, hwahae_col_map, 'hwahae')
df_coos_std   = standardize_schema(df_coos,   coos_col_map,   'coos')
df_paula_std  = standardize_schema(df_paula,  paula_col_map,  'paulaschoice')

print('스키마 표준화 완료')
print(f'  화해: {df_hwahae_std.shape}')
print(f'  Coos: {df_coos_std.shape}')
print(f"  Paula's: {df_paula_std.shape}")

---
## 7. 데이터 통합 (Merge)

### 📌 가이드라인
**방법 A — 단순 concat (소스별 행 유지):**  
각 소스의 행을 그대로 이어붙임. 같은 성분이 여러 소스에 중복 존재.  
→ 소스별 비교 분석, 성분 커버리지 확인에 적합

**방법 B — ingredient_key 기준 outer merge (성분 중심):**  
동일 성분을 한 행으로 합침. 소스별 정보가 컬럼으로 나뉨.  
→ 성분 정보 보완, 한영 매핑 테이블 생성에 적합

In [ ]:
# ─── 방법 A: 단순 concat ──────────────────────────────────────────
df_combined = pd.concat(
    [df_hwahae_std, df_coos_std, df_paula_std],
    ignore_index=True
)

print(f'통합 데이터셋 shape: {df_combined.shape}')
print(f"소스별 행 수:\n{df_combined['source'].value_counts()}")
df_combined.head()

In [ ]:
# ─── 방법 B: ingredient_key 기준 pivot (선택 사항) ─────────────────
# 각 소스에서 성분별 대표 정보 추출
def get_representative(df, source):
    return (
        df[df['source'] == source]
        .drop_duplicates('ingredient_key')
        .set_index('ingredient_key')
        .add_suffix(f'_{source}')
    )

rep_hwahae = get_representative(df_combined, 'hwahae')
rep_coos   = get_representative(df_combined, 'coos')
rep_paula  = get_representative(df_combined, 'paulaschoice')

df_merged = (
    rep_hwahae
    .join(rep_coos,   how='outer')
    .join(rep_paula,  how='outer')
    .reset_index()
)

# 성분명 한글/영문 통합 (coalesce 패턴)
name_ko_cols = [c for c in df_merged.columns if 'ingredient_name_ko' in c]
name_en_cols = [c for c in df_merged.columns if 'ingredient_name_en' in c]

if name_ko_cols:
    df_merged['ingredient_name_ko'] = df_merged[name_ko_cols].bfill(axis=1).iloc[:, 0]
if name_en_cols:
    df_merged['ingredient_name_en'] = df_merged[name_en_cols].bfill(axis=1).iloc[:, 0]

print(f'성분 중심 통합 데이터 shape: {df_merged.shape}')
df_merged.head()

---
## 8. 품질 검증 체크리스트

### 📌 가이드라인
저장 전 최종 품질 검증을 수행합니다.

In [ ]:
def quality_check(df, name='통합 데이터'):
    print(f'\n{'='*50}')
    print(f'🔍 품질 검증: {name}')
    print(f'{'='*50}')
    
    # 1. 기본 통계
    print(f'\n📐 Shape: {df.shape}')
    
    # 2. 결측률
    null_rate = (df.isnull().sum() / len(df) * 100).round(2)
    print(f'\n🔴 결측률 (5% 이상 경고):')
    print(null_rate[null_rate > 0].to_string() or '  결측 없음')
    
    # 3. ingredient_key 중복
    if 'ingredient_key' in df.columns:
        dup = df['ingredient_key'].duplicated().sum()
        status = '⚠️ 중복 있음' if dup > 0 else '✅ 중복 없음'
        print(f'\n🔑 ingredient_key 중복: {dup}건 {status}')
    
    # 4. 소스별 분포
    if 'source' in df.columns:
        print(f'\n📦 소스별 분포:')
        print(df['source'].value_counts().to_string())
    
    # 5. 빈 문자열 확인
    str_cols = df.select_dtypes('object').columns
    empty_counts = {col: (df[col] == '').sum() for col in str_cols if (df[col] == '').sum() > 0}
    if empty_counts:
        print(f'\n⚠️ 빈 문자열 (""): {empty_counts}')
    else:
        print('\n✅ 빈 문자열 없음')

quality_check(df_combined, '소스별 통합 (concat)')
quality_check(df_merged,   '성분 중심 통합 (merge)')

---
## 9. 저장

### 📌 가이드라인
- `utf-8-sig`: 한글 Excel 호환
- 원본 보존, 전처리 결과 별도 저장
- 파일명에 날짜 포함 권장

In [ ]:
from datetime import date

today = date.today().strftime('%Y%m%d')

# 소스별 통합 (방법 A)
out_combined = OUTPUT_DIR / f'ingredients_combined_{today}.csv'
df_combined.to_csv(out_combined, index=False, encoding='utf-8-sig')
print(f'✅ 저장 완료: {out_combined}  ({len(df_combined):,}행)')

# 성분 중심 통합 (방법 B)
out_merged = OUTPUT_DIR / f'ingredients_merged_{today}.csv'
df_merged.to_csv(out_merged, index=False, encoding='utf-8-sig')
print(f'✅ 저장 완료: {out_merged}  ({len(df_merged):,}행)')

print('\n🎉 전처리 파이프라인 완료!')

---
## 📋 전처리 체크리스트 요약

| # | 단계 | 상태 |
|---|---|---|
| 1 | 데이터 로드 (인코딩 처리) | ☐ |
| 2 | 기본 EDA (shape, 결측률, 유니크수) | ☐ |
| 3 | 결측치 처리 (핵심 컬럼 제거 / 텍스트 채움) | ☐ |
| 4 | 중복 제거 | ☐ |
| 5 | 성분명 정규화 (대소문자, 특수문자, 유니코드) | ☐ |
| 6 | 컬럼 스키마 통일 | ☐ |
| 7 | 소스 통합 (concat / merge 선택) | ☐ |
| 8 | 품질 검증 | ☐ |
| 9 | 결과 저장 (utf-8-sig) | ☐ |

> **⚠️ 주의사항:**  
> - 각 셀의 컬럼명(`'성분명'`, `'ingredient_name'` 등)은 실제 파일 컬럼명 확인 후 수정 필요  
> - `DATA_DIR` 경로를 실제 데이터 위치로 수정  
> - 정규화 규칙(슬래시 처리 등)은 도메인 지식에 따라 조정